# PPO from scratch — **V0: Vanilla Policy Gradient baseline**

This is the first of six notebooks (V0 → V5). Each adds **one** concept on top of the last,
and the final one (V5) is your own PPO, comparable to `train_and_record_ppo.py` on Acrobot.

| Version | Concept added |
|---|---|
| **V0 (this)** | Vanilla policy gradient (REINFORCE + reward-to-go) **+ a reusable eval harness** |
| V1 | Learned value baseline (a critic network) |
| V2 | Fixed-horizon rollout buffer + bootstrapping on cutoff |
| V3 | GAE-λ advantage |
| V4 | PPO-clip surrogate + multiple update passes |
| V5 | KL early-stop + advantage normalization → your full PPO |

### What V0 is for
You already have this algorithm working in `train_and_record.py`. Rebuilding it here
**from first principles** gives us (a) a clean baseline number on Acrobot to beat, and
(b) all the plumbing — env, seeding, evaluation, GIF — that V1–V5 reuse unchanged.

### The concepts *you* implement in V0
1. The **policy** as a `Categorical` distribution over actions.
2. **Reward-to-go**: each action is credited with the return that *follows* it.
3. The **REINFORCE loss**: `-(log π(a|s) · reward_to_go).mean()`.
4. The **training loop**: collect episodes → one gradient step → repeat.

Cells I've filled in for you: imports, config, `mlp`, the environment helpers, the
**evaluation + GIF harness**, and the **unit tests**. Cells marked `# TODO (V0)` are yours.

> **Kernel:** run this with the `ppo` conda env
> (`/home/slini/miniconda3/envs/ppo/bin/python`). gym 0.15.7 classic API.

### ✅ Done-when (V0 is finished)
- The `reward_to_go` unit-test cell passes.
- Training on `Acrobot-v1` runs and the printed return improves over the first epochs.
- The eval harness reports an average greedy return. Expect roughly **−100 to −150, but
  noisy** (e.g. ≈ −103 ± 70 over 10 episodes after ~30 epochs). Acrobot is quite learnable
  even with vanilla PG — so the story of V1–V5 is **less about final score and more about
  variance, stability, and sample-efficiency**: hitting a good policy faster, more reliably,
  and with a tighter spread. Watch that ± number shrink as much as the mean.


## Imports & configuration *(given)*

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.distributions.categorical import Categorical
import gymnasium as gym
from gymnasium.spaces import Discrete, Box

# ---- experiment configuration (shared by every version) --------------------
ENV_NAME   = "Acrobot-v1"
SEED       = 0
HIDDEN     = [64, 64]     # policy MLP hidden layers
LR         = 1e-2
EPOCHS     = 15
BATCH_SIZE = 4000         # min transitions collected per epoch
GAMMA      = 0.99         # reward-to-go discount (1.0 == undiscounted)

print("gymnasium", gym.__version__, "| torch", torch.__version__)

gymnasium 1.3.0 | torch 2.13.0+cpu


## Boilerplate: network + env helpers *(given)*

In [2]:
class MyPolicy(nn.Module):
    def __init__(self, input_size: int, output_size: int) -> None:
        super().__init__()
        sizes = [input_size] + HIDDEN + [output_size]
        layers = []
        for in_sz, out_sz in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_sz, out_sz))
            layers.append(nn.Tanh())
        layers = layers[:-1]
        self.linear_layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> Categorical:
        return Categorical(logits=self.linear_layers(x))
    
    def greedy(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_layers(x).argmax()

    def sample(self, x: torch.Tensor) -> torch.Tensor:
        return self(x).sample()


In [3]:



def make_env(env_name: str = ENV_NAME, render_mode: str | None = None) -> gym.Env:
    return gym.make(env_name, render_mode=render_mode)


def set_seed(env: gym.Env, seed: int = SEED) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    env.reset(seed=seed)
    env.action_space.seed(seed)


# quick look at the spaces we're dealing with
_e = make_env()
assert isinstance(_e.observation_space, Box)
assert isinstance(_e.action_space, Discrete)
OBS_DIM = _e.observation_space.shape[0]
N_ACTS  = _e.action_space.n
MAX_STEPS = _e.spec.max_episode_steps
print(f"{ENV_NAME}: obs_dim={OBS_DIM}, n_acts={N_ACTS}, max_steps={MAX_STEPS}")
_e.close()

Acrobot-v1: obs_dim=6, n_acts=3, max_steps=500


## 1. The policy *(you implement)*

A stochastic policy maps an observation to a **distribution over the 3 discrete actions**.
We represent it with a network that outputs one *logit* per action and wrap those logits in
`torch.distributions.Categorical`. That object gives you both `.sample()` (to act) and
`.log_prob(a)` (needed for the loss).

Implement:
- `policy_net`  — an `mlp` from `OBS_DIM` → `N_ACTS`.
- `get_policy(obs)` — returns a `Categorical` given a batch (or single) obs tensor.
- `get_action(obs)` — samples one action, returns a plain Python `int`.

In [4]:
x= torch.tensor(_e.reset()[0], dtype=torch.float32)
x.shape

torch.Size([6])

## 2. Reward-to-go *(you implement)*

Plain REINFORCE weights every action in an episode by the episode's *total* return. That's
noisy — an action shouldn't be credited for reward collected *before* it happened. **Reward-to-go**
fixes this: action at step *t* is weighted by the (discounted) sum of rewards from *t* onward:

$$R_t = \sum_{k=t}^{T} \gamma^{\,k-t}\, r_k$$

Implement `reward_to_go(rews, gamma)` returning a 1-D `np.ndarray` the same length as `rews`.
The unit-test cell below pins down the exact values it must produce.

In [5]:
def reward_to_go(rews: list[float], gamma: float = GAMMA) -> list[float]:
    """Discounted reward-to-go for one episode. rews: list/1-D array of floats."""
    n = len(rews)
    gamma_pows = gamma ** np.arange(n)
    cum_rews =[]
    for i in range(n):
        left = n-i
        cr = (rews[i:] * gamma_pows[:left]).sum()
        cum_rews.append(cr)
    return cum_rews

### 🔬 Unit test — `reward_to_go` *(given)*

In [6]:
def _test_reward_to_go() -> None:
    # undiscounted: [1,1,1,1] -> [4,3,2,1]
    np.testing.assert_allclose(reward_to_go([1, 1, 1, 1], gamma=1.0), [4, 3, 2, 1])
    # discounted: last element is just the final reward
    out = reward_to_go([1.0, 2.0, 3.0], gamma=0.9)
    expected = [1 + 0.9*2 + 0.81*3, 2 + 0.9*3, 3.0]
    np.testing.assert_allclose(out, expected, rtol=1e-6)
    assert np.asarray(out).shape == (3,), "must return same length as input"
    print("reward_to_go OK:", np.round(out, 3))

_test_reward_to_go()

reward_to_go OK: [5.23 4.7  3.  ]


## 3. The loss *(you implement)*

The policy-gradient loss is the negative mean of `log π(a|s)` weighted by reward-to-go.
Minimizing it with Adam performs gradient *ascent* on expected return.

$$L = -\frac{1}{N}\sum_i \log \pi(a_i\mid s_i)\; R_{t_i}$$

Implement `compute_loss(obs, act, weights)` — all three are tensors of matching length.

In [7]:
def compute_loss(policy: MyPolicy, obs: torch.Tensor, act: torch.Tensor,
                 weights: torch.Tensor) -> torch.Tensor:
    logp = policy(obs).log_prob(act)
    return - ((logp)*weights).mean()


## 4. Collect a batch & take one step *(you implement)*

The training loop for one epoch:
1. Reset the env; step through it with `get_action`, storing `obs`, `act`, `rew` each step.
2. When an episode ends (`done`), convert its rewards to **reward-to-go** and append to the
   batch weights; record the episode's total return and length; reset.
3. Stop once you've collected ≥ `BATCH_SIZE` transitions.
4. Build tensors, call `compute_loss`, `backward()`, and one `optimizer.step()`.

Return `(batch_rets, batch_lens)` (lists) so we can print progress.

> Tip: obs from gym is a float64 ndarray — cast with `torch.as_tensor(obs, dtype=torch.float32)`.
> Actions for `Categorical.log_prob` should be a `torch.int`/`long` tensor.

In [8]:
NUM_TRAJECTORIES = 32

def train_one_epoch(env: gym.Env) -> tuple[list[float], list[int]]:
    # TODO (V0): implement the 4 steps above.
    #   - accumulate batch_obs, batch_acts, batch_weights across episodes
    #   - per-episode: batch_weights += reward_to_go(ep_rews).tolist()
    #   - after >= BATCH_SIZE steps: one gradient step on compute_loss(...)
    #   - return (batch_rets, batch_lens)
    obs_batch =[]
    action_batch =[]
    rewards_batch =[]
    total_reward_per_ep=[]
    episodes_lens=[]
    for _ in range(NUM_TRAJECTORIES):
        rewards = []
        obs = torch.tensor(env.reset()[0], dtype = torch.float32)
        finished = False
        while not finished:
            with torch.no_grad():
                action = policy_net.sample(obs)
            next_obs, rew, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            obs_batch.append(obs)
            action_batch.append(action)
            rewards.append(rew)
            if done:
                timeout = len(rewards) == MAX_STEPS
                episodes_lens.append(len(rewards))
                break
            else:
                obs = torch.tensor(next_obs, dtype=torch.float32)
        acc_rewards = reward_to_go(rewards)
        rewards_batch += acc_rewards
        total_reward_per_ep.append(sum(rewards))
    obs_batch_tensor = torch.stack(obs_batch)
    actions_tensor = torch.tensor(action_batch)
    rewards_tensor = torch.tensor(rewards_batch)
    optimizer.zero_grad()
    loss = compute_loss(policy_net, obs_batch_tensor, actions_tensor, rewards_tensor )
    loss.backward()
    optimizer.step()
    return total_reward_per_ep, episodes_lens

        
    

## Evaluation & recording harness *(given — reused by every version)*

In [9]:

def greedy_action(obs: np.ndarray) -> torch.Tensor:
    with torch.no_grad():
        return policy_net.greedy(torch.tensor(obs, dtype=torch.float32))
    

def evaluate(env_name: str = ENV_NAME, n_episodes: int = 10, seed: int = 1000) -> tuple[float, float]:
    """Average greedy return over n_episodes. This is our comparison metric."""
    env = make_env(env_name)
    rets = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed + i)
        done, ep_ret = False, 0.0
        while not done:
            obs, r, terminated, truncated, _ = env.step(greedy_action(obs))
            done = terminated or truncated
            ep_ret += r
        rets.append(ep_ret)
    env.close()
    return float(np.mean(rets)), float(np.std(rets))


def record_gif(out_path: str, env_name: str = ENV_NAME, episodes: int = 3, fps: int = 15) -> None:
    """Greedy rollout -> GIF. Guarded so a headless render failure won't crash training."""
    try:
        import imageio
        env = make_env(env_name, render_mode="rgb_array")
        frames = []
        for _ in range(episodes):
            obs, done = env.reset()[0], False
            while not done:
                frames.append(env.render())
                obs, _, terminated, truncated, _ = env.step(greedy_action(obs))
                done = terminated or truncated
        env.close()
        imageio.mimsave(out_path, frames, fps=fps)
        print(f"saved {len(frames)} frames -> {out_path}")
    except Exception as ex:
        print("record_gif skipped (likely headless):", repr(ex))

## Train *(given — drives your code)*

In [10]:
env = make_env()
set_seed(env, SEED)
policy_net = MyPolicy(OBS_DIM, N_ACTS)
optimizer =  Adam(policy_net.parameters(), lr=LR)
for epoch in range(EPOCHS):
    batch_rets, batch_lens = train_one_epoch(env)
    print("epoch %3d \t return %f \t ep_len %6.1f" %
          (epoch, np.mean(batch_rets), np.mean(batch_lens)))
env.close()

epoch   0 	 return -494.656250 	 ep_len  494.7


epoch   1 	 return -343.156250 	 ep_len  344.1


epoch   2 	 return -170.968750 	 ep_len  172.0


epoch   3 	 return -127.312500 	 ep_len  128.3


epoch   4 	 return -110.906250 	 ep_len  111.9


...

epoch  10 	 return -96.312500 	 ep_len   97.3


epoch  11 	 return -112.062500 	 ep_len  113.1


epoch  12 	 return -118.531250 	 ep_len  119.5


epoch  13 	 return -109.593750 	 ep_len  110.6


epoch  14 	 return -122.437500 	 ep_len  123.4


## ✅ Verify V0 *(given)*

Run these once training has finished.

In [11]:
mean_ret, std_ret = evaluate(n_episodes=10)
print(f"V0 greedy return on {ENV_NAME}: {mean_ret:.1f} +/- {std_ret:.1f}")
print("Baseline established. Note BOTH the mean and the +/- spread -- V1-V5 should")
print("improve the spread (stability) as much as the mean.")

# optional: visual sanity check (safe to run; skips itself if headless)
record_gif("v0_baseline.gif")

V0 greedy return on Acrobot-v1: -421.4 +/- 157.2
Baseline established. Note BOTH the mean and the +/- spread -- V1-V5 should
improve the spread (stability) as much as the mean.


saved 1500 frames -> v0_baseline.gif


---
### When V0 is done
Ping me and I'll review your `train_one_epoch`, `reward_to_go`, and loss before generating
**V1**, which swaps the noisy raw reward-to-go weight for an **advantage computed against a
learned value baseline** — your first critic network.